# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [ ]:
from pathlib import Path
import importlib.util

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [ ]:
# Parâmetros importados do arquivo config.py
print("Filtrar por quantidade de alelos TCC2:.........................", parametros.filtarar_por_qte_de_alelos_tcc2)
print("Parâmetro de filtragem median binding percentile TCC2:.........", parametros.parametro_de_filtragem_mbp_tcc2)
print("Percentual de match mínimo TCC2:...............................", parametros.percent_match_minimo_tcc2)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 5
Percentual de match mínimo TCC1:............................... 95.0


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('./T CELL/DENV 3 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNGSWKLEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKLEKASLI,0.983079,0.02
1,1,QYKFQADSPKRLA,31,43,13,HLA-DRB3*01:01,7,0.05,FQADSPKRL,0.927520,0.05
2,1,KNGSWKLEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.06,WKLEKASLI,0.972226,0.06
3,1,QYKFQADSPKRLAT,31,44,14,HLA-DRB3*01:01,75,0.06,FQADSPKRL,0.921728,0.06
4,1,KNGSWKLEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.07,WKLEKASLI,0.950728,0.07
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,KLIHEWCCRSCTLPPLRYMGE,306,326,21,HLA-DRB3*02:02,603,100.00,CTLPPLRYM,0.000019,100.00
16412,1,KLIHEWCCRSCTLPPLRYMG,306,325,20,HLA-DRB1*04:01,536,100.00,CRSCTLPPL,0.000018,100.00
16413,1,KLIHEWCCRSCTLPPLRYMGE,306,326,21,HLA-DRB1*04:01,603,100.00,CRSCTLPPL,0.000015,100.00
16414,1,TTVSGKLIHEWCC,301,313,13,HLA-DRB3*02:02,61,100.00,VSGKLIHEW,0.000013,100.00


## Selecionando Epítopos por median binding percentile.

In [ ]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc2

In [ ]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  5


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2606,1,CLWPKTHTLW,223,232,10,HLA-B*44:03,567,4.90,CLWPKTHLW,CLWPKTHTLW,0.003115,4.90
2607,1,NELNYVLWE,73,81,9,HLA-B*44:03,73,4.90,NELNYVLWE,NELNYVLWE,0.003085,4.90
2608,1,DQKAVHADMGY,190,200,11,HLA-B*44:02,877,4.90,DQKAVHMGY,DQKAVHADMGY,0.002847,4.90
2609,1,TPPVSDLKY,105,113,9,HLA-B*44:02,105,4.90,TPPVSDLKY,TPPVSDLKY,0.002830,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [ ]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAVKDERAVHADM,186,198,1,3.600,HLA-DRB1*01:01
1,AAVKDERAVHADMG,186,199,1,4.100,HLA-DRB1*01:01
2,AAVKDERAVHADMGYWIE,186,203,1,4.700,HLA-DRB3*01:01
3,AAVKDERAVHADMGYWIES,186,204,1,3.300,HLA-DRB3*01:01
4,AAVKDERAVHADMGYWIESQ,186,205,1,2.300,HLA-DRB3*01:01
...,...,...,...,...,...,...
220,YRPGYHTQTAGPWHLGK,256,272,4,2.150,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
221,YRPGYHTQTAGPWHLGKL,256,273,4,3.050,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*04:01/DQB1..."
222,YRPGYHTQTAGPWHLGKLE,256,274,2,1.845,"HLA-DQA1*05:01/DQB1*02:01, HLA-DQA1*05:01/DQB1..."
223,YRPGYHTQTAGPWHLGKLEL,256,275,2,2.125,"HLA-DQA1*05:01/DQB1*02:01, HLA-DQA1*05:01/DQB1..."


## Filtragem por qte_de_alelos

In [ ]:
# qte_de_alelos_minima = parametros.filtarar_por_qte_de_alelos_tcc2
qte_de_alelos_minima = 2

In [ ]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_de_alelos_minima
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ATRLENIMW,60,68,13,3.300,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
2,CIWPKSHTL,223,231,20,1.600,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
3,CTLPPLRFK,316,324,12,1.850,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
4,ECPDNQRAW,142,150,11,1.900,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
5,ESEMIIPKIY,238,247,10,2.550,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*3..."
6,ETWKLARASF,208,217,13,2.800,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
7,EVHTWTEQY,24,32,16,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
8,EVHTWTEQYKF,24,34,10,3.250,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
9,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."


In [ ]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YGMEIRPISEKEENMVK,331,347,1,0.23,HLA-DQA1*05:01/DQB1*02:01
1,YGMEIRPISEKEENMVKSL,331,349,1,0.26,HLA-DQA1*05:01/DQB1*02:01
2,YGMEIRPISEKEENMVKSLV,331,350,1,0.35,HLA-DQA1*05:01/DQB1*02:01
3,YGMEIRPISEKEENMVKSLVS,331,351,1,0.38,HLA-DQA1*05:01/DQB1*02:01
4,GVFTTNIWLKLRE,161,173,5,0.40,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
...,...,...,...,...,...,...
220,VLESDMIIPKSLA,236,248,1,4.80,HLA-DRB1*03:01
221,VTNEVHTWTEQYKFQ,21,35,1,4.80,HLA-DQA1*04:01/DQB1*04:02
222,HRLMSAAVKDERAVHADM,181,198,1,4.90,HLA-DQA1*01:02/DQB1*06:02
223,SLAGPISQHNYRPGYH,246,261,1,4.90,HLA-DPA1*01:03/DPB1*02:01


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [ ]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_2.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0          YGMEIRPISEKEENMVK
1        YGMEIRPISEKEENMVKSL
2       YGMEIRPISEKEENMVKSLV
3      YGMEIRPISEKEENMVKSLVS
4              GVFTTNIWLKLRE
               ...          
220            VLESDMIIPKSLA
221          VTNEVHTWTEQYKFQ
222       HRLMSAAVKDERAVHADM
223         SLAGPISQHNYRPGYH
224        VTNEVHTWTEQYKFQAD
Name: peptide, Length: 225, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [ ]:
!seqkit grep -s -v -r -p '[-*X]' './Fastas/denv3_NS1_proteinas.fasta' > DENV3_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [ ]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,GVFTTNIWLKLRE,13,93.57% (597/638),84.62%,100.00%,NaN
1,2,NP 2,HTWTEQYKFQADSPKRL,17,94.67% (604/638),94.12%,100.00%,NaN
2,3,NP 3,GVFTTNIWLKLREV,14,72.88% (465/638),78.57%,100.00%,NaN
3,4,NP 4,QYKFQADSPKRLATAIAGA,19,94.04% (600/638),73.68%,100.00%,NaN
4,5,NP 5,HTWTEQYKFQADSPKRLAT,19,94.51% (603/638),84.21%,100.00%,NaN
...,...,...,...,...,...,...,...,...
107,108,NP 108,EDGCWYGMEIRPISEK,16,61.13% (390/638),87.50%,100.00%,NaN
108,109,NP 109,HRLMSAAVKDERAVH,15,80.88% (516/638),80.00%,100.00%,NaN
109,110,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,NaN
110,111,NP 111,LKYSWKTWGKAKIVTAETQ,19,94.51% (603/638),68.42%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [ ]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,GVFTTNIWLKLRE,13,93.57% (597/638),84.62%,100.00%,5,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
1,NP 2,HTWTEQYKFQADSPKRL,17,94.67% (604/638),94.12%,100.00%,2,"HLA-DRB3*01:01, HLA-DRB5*01:01"
2,NP 3,GVFTTNIWLKLREV,14,72.88% (465/638),78.57%,100.00%,4,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
3,NP 4,QYKFQADSPKRLATAIAGA,19,94.04% (600/638),73.68%,100.00%,2,"HLA-DRB3*01:01, HLA-DRB3*02:02"
4,NP 5,HTWTEQYKFQADSPKRLAT,19,94.51% (603/638),84.21%,100.00%,4,"HLA-DRB1*04:01, HLA-DRB3*01:01, HLA-DRB3*02:02..."
...,...,...,...,...,...,...,...,...
107,NP 108,EDGCWYGMEIRPISEK,16,61.13% (390/638),87.50%,100.00%,2,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*11:01"
108,NP 109,HRLMSAAVKDERAVH,15,80.88% (516/638),80.00%,100.00%,2,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*03:01"
109,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,3,"HLA-DQA1*01:01/DQB1*05:01, HLA-DRB1*08:02, HLA..."
110,NP 111,LKYSWKTWGKAKIVTAETQ,19,94.51% (603/638),68.42%,100.00%,2,"HLA-DQA1*05:01/DQB1*03:01, HLA-DRB1*13:02"


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [ ]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,GVFTTNIWLKLRE,13,93.57% (597/638),84.62%,100.00%,5,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",93.57
1,NP 2,HTWTEQYKFQADSPKRL,17,94.67% (604/638),94.12%,100.00%,2,"HLA-DRB3*01:01, HLA-DRB5*01:01",94.67
2,NP 3,GVFTTNIWLKLREV,14,72.88% (465/638),78.57%,100.00%,4,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",72.88
3,NP 4,QYKFQADSPKRLATAIAGA,19,94.04% (600/638),73.68%,100.00%,2,"HLA-DRB3*01:01, HLA-DRB3*02:02",94.04
4,NP 5,HTWTEQYKFQADSPKRLAT,19,94.51% (603/638),84.21%,100.00%,4,"HLA-DRB1*04:01, HLA-DRB3*01:01, HLA-DRB3*02:02...",94.51
...,...,...,...,...,...,...,...,...,...
107,NP 108,EDGCWYGMEIRPISEK,16,61.13% (390/638),87.50%,100.00%,2,"HLA-DQA1*04:01/DQB1*04:02, HLA-DRB1*11:01",61.13
108,NP 109,HRLMSAAVKDERAVH,15,80.88% (516/638),80.00%,100.00%,2,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*03:01",80.88
109,NP 110,VTNEVHTWTEQYKFQADSPK,20,98.75% (630/638),95.00%,100.00%,3,"HLA-DQA1*01:01/DQB1*05:01, HLA-DRB1*08:02, HLA...",98.75
110,NP 111,LKYSWKTWGKAKIVTAETQ,19,94.51% (603/638),68.42%,100.00%,2,"HLA-DQA1*05:01/DQB1*03:01, HLA-DRB1*13:02",94.51


### Sort e filtragem por percent_match

In [ ]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= parametros.percent_match_minimo_tcc2]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 64,QYKFQPESPSKLAS,14,99.19% (859/866),92.86%,100.00%,7,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
1,NP 97,QYKFQPESPSKLA,13,99.19% (859/866),92.31%,100.00%,9,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
2,NP 47,QYKFQPESPSKLASA,15,99.19% (859/866),93.33%,100.00%,7,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
3,NP 107,HTWTEQYKFQPESPSKLASA,20,99.19% (859/866),95.00%,100.00%,6,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",99.19
4,NP 137,HTWTEQYKFQPESPSKLAS,19,99.19% (859/866),94.74%,100.00%,6,"HLA-DRB1*01:01, HLA-DRB1*04:01, HLA-DRB1*04:05...",99.19
5,NP 85,QYKFQPESPSKLASAI,16,98.96% (857/866),93.75%,100.00%,5,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",98.96
6,NP 125,HTWTEQYKFQPESPSKLASAI,21,98.96% (857/866),95.24%,100.00%,6,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",98.96
7,NP 119,ADMGYWIESALNDTWK,16,97.23% (842/866),68.75%,100.00%,7,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1...",97.23
8,NP 126,ADMGYWIESALNDT,14,97.23% (842/866),71.43%,100.00%,5,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1...",97.23
9,NP 98,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,7,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",97.00
